<a href="https://colab.research.google.com/github/bradtanguin/cod-prediction-process-stabilization/blob/main/cod_data_informatics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

# Visualization

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Warnings

import warnings
warnings.filterwarnings('ignore')

# time-series
from statsmodels.graphics.tsaplots import month_plot, quarter_plot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_excel('/content/drive/MyDrive/COD_ML_Project/Data/wrf_historical_data.xlsx',
                   parse_dates = True
                   ).set_index('Datetime')

In [4]:
df.columns

Index(['IN, PH', 'IN, NO3-N', 'IN, Fecal Coli', 'IN, BOD', 'IN, COD',
       'IN, Oil & Grease', 'IN, PO4-P', 'IN, Surfactants (MBAS)', 'IN, TSS',
       'IN, VSS', 'IN, NH3-N', 'IN, Flow (m³/hr)', 'IN, Totalizer', 'EFF, PH',
       'EFF, NO3-N', 'EFF, Fecal Coli', 'EFF, BOD', 'EFF, COD',
       'EFF, Oil & Grease', 'EFF, PO4-P', 'EFF, Surfactants (MBAS)',
       'EFF, TSS', 'EFF, VSS', 'EFF, NH3-N', 'EFF, Flow (m³/hr)',
       'EFF, Totalizer', 'B1A, ORP', 'B1A, Water Level', 'B1AX, ORP',
       'B1AX, Water Level', 'B1O, MLSS', 'B1O, Temperature', 'B1O, DO',
       'B1O, SV30', 'B1O, SVl', 'B1O, Gravimetric', 'B1O, Tank LIT Reading',
       'C1, Overflow Status', 'C1, Center Drive', 'C2, Overflow Status',
       'C2, Center Drive', 'RES, Volume Wasted', 'RES, MLSS', 'RES, Totalizer',
       'Overall Electric Consumption'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1188 entries, 2023-03-01 to 2026-05-31
Data columns (total 45 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   IN, PH                        848 non-null    float64
 1   IN, NO3-N                     848 non-null    float64
 2   IN, Fecal Coli                848 non-null    float64
 3   IN, BOD                       848 non-null    float64
 4   IN, COD                       848 non-null    float64
 5   IN, Oil & Grease              170 non-null    float64
 6   IN, PO4-P                     848 non-null    float64
 7   IN, Surfactants (MBAS)        170 non-null    float64
 8   IN, TSS                       848 non-null    float64
 9   IN, VSS                       848 non-null    float64
 10  IN, NH3-N                     848 non-null    float64
 11  IN, Flow (m³/hr)              1188 non-null   float64
 12  IN, Totalizer                 848 non-null  

# Dropping Columns

*   Since the objective of this study is to predict influent COD, parameters related to the effluent, biotank, clarifier, and RES wasting were excluded, as they do not directly influence the characteristics of the raw wastewater influent.

*   The totalizer feature was also exluded since the influent flow rate already gives the idea of the hydraulic condition.


In [6]:
columns_to_drop = [
    'IN, Totalizer',
    'EFF, PH',
    'EFF, NO3-N',
    'EFF, Fecal Coli',
    'EFF, BOD',
    'EFF, COD',
    'EFF, Oil & Grease',
    'EFF, PO4-P',
    'EFF, Surfactants (MBAS)',
    'EFF, TSS',
    'EFF, VSS',
    'EFF, NH3-N',
    'EFF, Flow (m³/hr)',
    'EFF, Totalizer',
    'B1A, ORP',
    'B1A, Water Level',
    'B1AX, ORP',
    'B1AX, Water Level',
    'B1O, MLSS',
    'B1O, Temperature',
    'B1O, DO',
    'B1O, SV30',
    'B1O, SVl',
    'B1O, Gravimetric',
    'B1O, Tank LIT Reading',
    'C1, Overflow Status',
    'C1, Center Drive',
    'C2, Overflow Status',
    'C2, Center Drive',
    'RES, Volume Wasted',
    'RES, MLSS',
    'RES, Totalizer',
    'Overall Electric Consumption'
]
df = df.drop(columns=columns_to_drop)
display(df.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1188 entries, 2023-03-01 to 2026-05-31
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   IN, PH                  848 non-null    float64
 1   IN, NO3-N               848 non-null    float64
 2   IN, Fecal Coli          848 non-null    float64
 3   IN, BOD                 848 non-null    float64
 4   IN, COD                 848 non-null    float64
 5   IN, Oil & Grease        170 non-null    float64
 6   IN, PO4-P               848 non-null    float64
 7   IN, Surfactants (MBAS)  170 non-null    float64
 8   IN, TSS                 848 non-null    float64
 9   IN, VSS                 848 non-null    float64
 10  IN, NH3-N               848 non-null    float64
 11  IN, Flow (m³/hr)        1188 non-null   float64
dtypes: float64(12)
memory usage: 120.7 KB


None

# Frequency

In [7]:
# Setting Frequency to daily

df = df.asfreq('D')

df_freq= df.index.freq

print(f'The Frequency of the dataframe is: {df_freq}')

The Frequency of the dataframe is: <Day>


In [8]:
df.describe(include='all')

,"IN, PH","IN, NO3-N","IN, Fecal Coli","IN, BOD","IN, COD","IN, Oil & Grease","IN, PO4-P","IN, Surfactants (MBAS)","IN, TSS","IN, VSS","IN, NH3-N","IN, Flow (m³/hr)"
count,848.000000,848.000000,8.480000e+02,848.000000,848.000000,170.000000,848.000000,170.000000,848.000000,848.000000,848.000000,1188.000000
mean,7.104610,1.048149,1.250888e+06,70.817099,158.949888,4.917882,3.196934,1.556059,115.399057,86.699882,15.593703,2168.989394
std,0.359993,0.543916,4.342063e+05,9.606286,18.823251,1.220157,0.522541,0.292877,17.140428,13.505026,2.742558,159.203689
min,6.100203,0.101000,4.940320e+05,42.900000,101.734993,3.000000,2.000000,0.940000,68.100000,51.400000,10.000000,1901.500000
25%,6.858205,0.599500,8.730142e+05,64.375000,147.392608,4.015000,2.840000,1.350000,103.500000,77.000000,13.667500,2045.025000
50%,7.094446,1.030000,1.266030e+06,70.600000,159.242839,4.925000,3.160000,1.520000,115.600000,86.750000,15.480000,2149.050000
75%,7.366097,1.519750,1.643346e+06,76.900000,170.148606,5.920000,3.560000,1.777500,126.000000,95.225000,17.372500,2288.500000
max,8.240547,1.994000,2.056457e+06,117.600000,245.100000,8.060000,5.040000,2.230000,178.300000,137.800000,28.600000,2508.100000


# Null Values

- According to the monitoring requirements, laboratory sampling and analysis were conducted four times per week, supplemented by one weekly grab sample analyzed by a third-party laboratory. Therefore, a total of five sampling data points were collected each week.


- The parameters that were only analyzed once per week through third-party laboratory testing are as follows:

    1.   Surfactants (MBAS)
    2.   IN, Oil & Grease

In [9]:
df_nulls = df.isnull().sum()
df_nulls.rename('Sum of Null Values', inplace=True)

df_nulls_percentage = (df.isnull().sum() / len(df)) * 100
df_nulls_percentage.rename('Percentage of Null Values', inplace=True)

null_df = pd.concat([df_nulls, df_nulls_percentage], axis=1)
null_df['Percentage of Null Values'] = null_df['Percentage of Null Values'].round(2)

display(null_df)

,Sum of Null Values,Percentage of Null Values
"IN, PH",340,28.62
"IN, NO3-N",340,28.62
"IN, Fecal Coli",340,28.62
"IN, BOD",340,28.62
"IN, COD",340,28.62
"IN, Oil & Grease",1018,85.69
"IN, PO4-P",340,28.62
"IN, Surfactants (MBAS)",1018,85.69
"IN, TSS",340,28.62
"IN, VSS",340,28.62


In [10]:
df.isnull().sum()*100/len(df)

,0
"IN, PH",28.619529
"IN, NO3-N",28.619529
"IN, Fecal Coli",28.619529
"IN, BOD",28.619529
"IN, COD",28.619529
"IN, Oil & Grease",85.690236
"IN, PO4-P",28.619529
"IN, Surfactants (MBAS)",85.690236
"IN, TSS",28.619529
"IN, VSS",28.619529


# Handling NAN Values

*  Dropping missing values was not performed because they constitute approximately 30% of the entire dataset, and removing them would result in a substantial loss of information.
---


- Appropriate Analysis in this study are as follows:


    1.   Forward Fill
    2.   Backward Fill
    3.   Time Interpolation

- Model performance was evaluated using Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), Mean Absolute Percentage Error (MAPE), and the coefficient of determination (R²). The imputation method that achieved the best predictive performance was selected for the final model development.

In [11]:
# ffill method

print('ffill Method')
ffill_df = df.ffill()
display(ffill_df.isnull().sum())
print('--'*20)

#bfill method
print('bfill Method')
bfill_df = df.bfill()
display(bfill_df.isnull().sum())
print('--'*20)

# Time Interpolation Method
print('Time Interpolation Method')
interpolation_df = df.interpolate(method='time')
display(interpolation_df.isnull().sum())

ffill Method


,0
"IN, PH",0
"IN, NO3-N",0
"IN, Fecal Coli",0
"IN, BOD",0
"IN, COD",0
"IN, Oil & Grease",1
"IN, PO4-P",0
"IN, Surfactants (MBAS)",1
"IN, TSS",0
"IN, VSS",0


----------------------------------------
bfill Method


,0
"IN, PH",2
"IN, NO3-N",2
"IN, Fecal Coli",2
"IN, BOD",2
"IN, COD",2
"IN, Oil & Grease",3
"IN, PO4-P",2
"IN, Surfactants (MBAS)",3
"IN, TSS",2
"IN, VSS",2


----------------------------------------
Time Interpolation Method


,0
"IN, PH",0
"IN, NO3-N",0
"IN, Fecal Coli",0
"IN, BOD",0
"IN, COD",0
"IN, Oil & Grease",1
"IN, PO4-P",0
"IN, Surfactants (MBAS)",1
"IN, TSS",0
"IN, VSS",0
